# CWA vs SELU vs GELU: Char-GPT on Tiny Shakespeare

This notebook demonstrates the **Curie-Weiss Activation (CWA)** — a novel activation function with a learnable scalar coupling parameter `J` that implements a mean-field fixed-point equation within each forward pass:

$$m^* = \text{mean}(\tanh(x + J \cdot m^*))$$

The backward pass uses a closed-form Implicit Function Theorem (IFT) gradient in O(n) time:

$$\frac{\partial L}{\partial x_i} = s^2_i \left(g_i + \frac{J \sum_k g_k s^2_k}{n(1 - J\bar{s})}\right)$$

**Experiment**: A 6-layer char-GPT is trained on Tiny Shakespeare and we compare:
1. **CWA** — Novel mean-field activation with learnable coupling
2. **SELU** — Self-normalizing baseline with LeCun initialization
3. **GELU** — Standard transformer reference

The notebook loads pre-computed reference results from `mini_demo_data.json` and runs a minimal live training demo.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru and psutil are NOT pre-installed on Colab
_pip('loguru==0.7.3', 'psutil==6.1.1')

# Core packages: pre-installed on Colab, install locally to match
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0', 'torch==2.9.0')

In [ ]:
import math
import json
import os
import sys
import time
import urllib.request
import gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import psutil
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-2f6f35-curie-weiss-activation-formal-verificati/main/round-5/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset: {data['metadata']['method']}")
print(f"Total examples: {data['metadata']['total_examples']}")
print(f"Pre-computed run config: {data['metadata']['config']}")

## Configuration

These parameters control the **live demo training run**. They are set to small values for a fast demo.
The original paper used `n_steps=5000`, `n_embd=256`, `n_layer=6`, `seq_len=256`, `batch=64`.
To reproduce full results, comment out the demo values and uncomment the originals.

In [ ]:
# ── Demo config (fast, works on CPU) ──────────────────────────────────────────
N_EMBD       = 64       # original: 256
N_HEAD       = 4        # original: 8
N_LAYER      = 2        # original: 6
SEQ_LEN      = 64       # original: 256
BATCH        = 8        # original: 64
LR           = 3e-4     # same as original
N_STEPS      = 100      # original: 5000
EVAL_INTERVAL = 20      # original: 100
EVAL_ITERS   = 3        # original: 50
WARMUP       = 10       # original: min(100, n_steps//5)
SEEDS        = [42]     # original: [42, 7]
ACTIVATIONS  = ["gelu", "selu", "cwa"]

## Hardware Detection

Detects available compute (GPU/CPU) and sets RAM limits.

In [ ]:
import resource

def _detect_cpus() -> int:
    try:
        parts = Path("/sys/fs/cgroup/cpu.max").read_text().split()
        if parts[0] != "max":
            return math.ceil(int(parts[0]) / int(parts[1]))
    except (FileNotFoundError, ValueError):
        pass
    try:
        q = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read_text())
        p = int(Path("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read_text())
        if q > 0:
            return math.ceil(q / p)
    except (FileNotFoundError, ValueError):
        pass
    try:
        return len(os.sched_getaffinity(0))
    except (AttributeError, OSError):
        pass
    return os.cpu_count() or 1


def _container_ram_gb() -> float | None:
    for p in ["/sys/fs/cgroup/memory.max", "/sys/fs/cgroup/memory/memory.limit_in_bytes"]:
        try:
            v = Path(p).read_text().strip()
            if v != "max" and int(v) < 1_000_000_000_000:
                return int(v) / 1e9
        except (FileNotFoundError, ValueError):
            pass
    return None


NUM_CPUS = _detect_cpus()
HAS_GPU = torch.cuda.is_available()
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if HAS_GPU else 0
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
TOTAL_RAM_GB = _container_ram_gb() or psutil.virtual_memory().total / 1e9

logger.info(f"Hardware: CPUs={NUM_CPUS}, GPU={HAS_GPU}, VRAM={VRAM_GB:.1f}GB, RAM={TOTAL_RAM_GB:.1f}GB, device={DEVICE}")

# Set memory limits (use 22GB of 28GB container)
RAM_BUDGET = int(22 * 1024**3)
try:
    resource.setrlimit(resource.RLIMIT_AS, (RAM_BUDGET, RAM_BUDGET))
    logger.info(f"RAM limit set to 22GB")
except ValueError:
    logger.warning("Could not set RAM limit (may already be set lower)")

if HAS_GPU:
    _free, _total = torch.cuda.mem_get_info(0)
    torch.cuda.set_per_process_memory_fraction(0.85)
    logger.info(f"VRAM budget: 85% of {_total/1e9:.1f}GB")

## Dataset: Tiny Shakespeare

Downloads Tiny Shakespeare (~1M characters) and creates character-level train/val splits.
The vocabulary consists of all unique characters in the text.

In [ ]:
WORKSPACE = Path(".")
DATA_FILE = WORKSPACE / "input.txt"
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def load_dataset() -> tuple[torch.Tensor, torch.Tensor, dict, int]:
    if not DATA_FILE.exists():
        logger.info(f"Downloading Tiny Shakespeare from {URL}")
        try:
            urllib.request.urlretrieve(URL, str(DATA_FILE))
            logger.info("Download complete")
        except Exception as e:
            logger.warning(f"Download failed ({e}), trying fallback via HuggingFace datasets")
            try:
                from datasets import load_dataset as hf_load
                ds = hf_load("tiny_shakespeare")
                text = "\n".join(ds["train"]["text"] + ds["validation"]["text"] + ds["test"]["text"])
                DATA_FILE.write_text(text)
                logger.info("HuggingFace fallback succeeded")
            except Exception as e2:
                logger.error(f"All download attempts failed: {e2}")
                raise

    text = DATA_FILE.read_text()
    logger.info(f"Dataset: {len(text):,} characters")

    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for i, c in enumerate(chars)}
    logger.info(f"Vocab size: {vocab_size}")

    data_tensor = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    n = int(0.9 * len(data_tensor))
    train_data = data_tensor[:n]
    val_data = data_tensor[n:]
    logger.info(f"Train: {len(train_data):,} tokens, Val: {len(val_data):,} tokens")
    return train_data, val_data, itos, vocab_size


def get_batch(split: str, train_data: torch.Tensor, val_data: torch.Tensor,
              seq_len: int, batch: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    data_t = train_data if split == "train" else val_data
    ix = torch.randint(len(data_t) - seq_len, (batch,))
    x = torch.stack([data_t[i:i + seq_len] for i in ix])
    y = torch.stack([data_t[i + 1:i + seq_len + 1] for i in ix])
    return x.to(device), y.to(device)


train_data, val_data, itos, vocab_size = load_dataset()

## CWA: Curie-Weiss Activation

The core innovation. The activation implements a mean-field fixed-point equation:
$$m^* = \text{mean}(\tanh(x + J \cdot m^*))$$

**Forward**: iterates to find $m^*$ using warm-start + adaptive convergence.

**Backward**: uses closed-form IFT gradient — no autograd through the iteration:
$$\frac{\partial L}{\partial x_i} = s^2_i \left(g_i + \frac{J \sum_k g_k s^2_k}{n(1 - J\bar{s})}\right)$$
where $s^2_i = \text{sech}^2(x_i + J m^*)$ and $\bar{s} = \text{mean}(s^2)$.

In [ ]:
class CWAFunction(torch.autograd.Function):
    """Curie-Weiss Activation with closed-form IFT backward.

    Fixed-point equation: m* = mean(tanh(x + J*m*))  [scalar mean-field]
    IFT gives closed-form: grad_x_i = s2_i*(grad_i + J*sum_gs2/(n*(1-J*s_bar)))
    """

    @staticmethod
    def forward(ctx, x: torch.Tensor, J_raw: torch.Tensor, K_max: int, warm_T: int) -> torch.Tensor:
        J = torch.sigmoid(J_raw)
        B, T, H = x.shape

        # Warm start (no grad)
        m = x.new_zeros(B, T, 1)
        with torch.no_grad():
            for _ in range(warm_T):
                m = torch.tanh(x + J * m).mean(dim=-1, keepdim=True)

            # Adaptive delta based on current J·s̄
            s2_tmp = torch.cosh(x + J * m).pow(-2)
            s_bar_tmp = s2_tmp.mean(dim=-1, keepdim=True)
            J_s_bar_scalar = (J * s_bar_tmp.mean()).item()
            delta = max(1e-4 * max(1 - J_s_bar_scalar, 1e-3), 1e-7)

            # Iterate to convergence
            k_used = K_max
            for k in range(K_max):
                m_new = torch.tanh(x + J * m).mean(dim=-1, keepdim=True)
                if (m_new - m).abs().max().item() < delta:
                    m = m_new
                    k_used = k + 1
                    break
                m = m_new

            # Final s2, s_bar at converged m*
            s2 = torch.cosh(x + J * m).pow(-2)
            s_bar = s2.mean(dim=-1, keepdim=True)
            J_s_bar = J * s_bar

        y = torch.tanh(x + J * m)
        ctx.save_for_backward(x, J_raw, m, s2, s_bar, J_s_bar)
        ctx.k_used = k_used
        return y

    @staticmethod
    def backward(ctx, grad_y: torch.Tensor):
        x, J_raw, m, s2, s_bar, J_s_bar = ctx.saved_tensors
        J = torch.sigmoid(J_raw)
        n = x.shape[-1]

        # Clamp denominator to avoid instability
        denom = (1.0 - J_s_bar).clamp(min=1e-6)

        # IFT gradient: ∂L/∂x_i = s2_i*(grad_i + J*sum(grad_k*s2_k)/(n*denom))
        sum_g_s2 = (grad_y * s2).sum(dim=-1, keepdim=True)
        grad_x = s2 * (grad_y + J * sum_g_s2 / (n * denom))

        # ∂L/∂J: chain via dJ/dJ_raw = J*(1-J)
        grad_J = (grad_y * s2 * m / denom).sum()
        grad_J_raw = grad_J * J * (1.0 - J)

        return grad_x, grad_J_raw, None, None


class CWAActivation(nn.Module):
    """Curie-Weiss Activation: learnable scalar coupling J with IFT backward."""

    def __init__(self, K_max: int = 50, warm_T: int = 3):
        super().__init__()
        self.J_raw = nn.Parameter(torch.zeros(1))  # J = sigmoid(0) = 0.5
        self.K_max = K_max
        self.warm_T = warm_T
        self.last_diag: dict = {}

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = CWAFunction.apply(x, self.J_raw, self.K_max, self.warm_T)

        # Collect diagnostics (cheap: reuse last converged state)
        with torch.no_grad():
            J = torch.sigmoid(self.J_raw)
            m = x.new_zeros(*x.shape[:-1], 1)
            for _ in range(self.warm_T + 5):
                m = torch.tanh(x + J * m).mean(dim=-1, keepdim=True)
            s2 = torch.cosh(x + J * m).pow(-2)
            s_bar = s2.mean(dim=-1, keepdim=True)
            self.last_diag = {
                "mean_act_mag": (x + J * m).abs().mean().item(),
                "mean_sech2": s2.mean().item(),
                "J_s_bar": (J * s_bar.mean()).item(),
                "J": J.item(),
            }
        return y

## SELU with LeCun Initialization

SELU requires LeCun normal init (std=1/sqrt(fan_in)) on the FFN linear layers to achieve
its self-normalizing property. Standard GPT init (std=0.02) is used for all other layers.

In [ ]:
def lecun_normal_init(module: nn.Module) -> None:
    """Apply LeCun normal init (std=1/sqrt(fan_in)) to all Linear layers."""
    for m in module.modules():
        if isinstance(m, nn.Linear):
            fan_in = m.weight.size(1)
            nn.init.normal_(m.weight, 0.0, 1.0 / math.sqrt(fan_in))
            if m.bias is not None:
                nn.init.zeros_(m.bias)

## GPT Architecture

Standard 6-layer causal transformer. The activation function is swapped per-run:
- `CWAActivation` for CWA
- `nn.SELU()` for SELU
- `nn.GELU()` for GELU

The MLP follows the standard GPT-2 4x expansion: `n_embd → 4*n_embd → n_embd`.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, seq_len: int):
        super().__init__()
        assert n_embd % n_head == 0
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.n_head = n_head
        self.n_embd = n_embd
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(seq_len, seq_len)).view(1, 1, seq_len, seq_len),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hs = C // self.n_head
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(hs))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class MLP(nn.Module):
    def __init__(self, n_embd: int, act_type: str):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.act_type = act_type
        if act_type == "cwa":
            self.act = CWAActivation(K_max=50, warm_T=3)
        elif act_type == "selu":
            self.act = nn.SELU()
        else:  # gelu
            self.act = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.act(self.fc1(x)))


class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, seq_len: int, act_type: str):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, seq_len)
        self.mlp = MLP(n_embd, act_type)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 256,
        n_head: int = 8,
        n_layer: int = 6,
        seq_len: int = 256,
        act_type: str = "gelu",
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, seq_len, act_type) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.seq_len = seq_len
        self.act_type = act_type

        self.apply(self._init_weights)
        if act_type == "selu":
            # Override FFN linears with LeCun normal init (applied after default init)
            for block in self.blocks:
                lecun_normal_init(block.mlp)

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def get_cwa_diagnostics(self) -> dict:
        diags = [
            block.mlp.act.last_diag
            for block in self.blocks
            if isinstance(block.mlp.act, CWAActivation) and block.mlp.act.last_diag
        ]
        if not diags:
            return {}
        return {k: sum(d[k] for d in diags) / len(diags) for k in diags[0]}

## Training Loop

Cosine LR schedule with linear warmup. Evaluation uses `eval_iters` batches from the
validation set to estimate BPC (bits per character = cross-entropy loss / log(2)).

For CWA, extra diagnostic arrays track:
- `J`: learned coupling parameter
- `J_s_bar`: mean-field susceptibility proxy (J × mean(sech²))
- `mean_act_mag`: mean absolute activation magnitude
- `mean_sech2`: mean sech² (saturation indicator)

In [ ]:
def get_lr(step: int, n_steps: int, lr: float, warmup: int) -> float:
    if step < warmup:
        return lr * step / max(warmup, 1)
    progress = (step - warmup) / max(n_steps - warmup, 1)
    return lr * 0.5 * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def estimate_val_loss(
    model: CharGPT,
    train_data: torch.Tensor,
    val_data: torch.Tensor,
    eval_iters: int,
    seq_len: int,
    batch: int,
    device: torch.device,
) -> float:
    model.eval()
    losses = []
    for _ in range(eval_iters):
        xb, yb = get_batch("val", train_data, val_data, seq_len, batch, device)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def train_run(
    act_type: str,
    seed: int,
    vocab_size: int,
    train_data: torch.Tensor,
    val_data: torch.Tensor,
    config: dict,
    device: torch.device,
) -> dict:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    logger.info(f"Starting {act_type} seed={seed}")
    model = CharGPT(
        vocab_size=vocab_size,
        n_embd=config["n_embd"],
        n_head=config["n_head"],
        n_layer=config["n_layer"],
        seq_len=config["seq_len"],
        act_type=act_type,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    logger.info(f"{act_type} seed={seed}: {n_params:,} parameters")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        betas=(0.9, 0.99),
        weight_decay=0.1,
    )

    val_bpc_history = []
    cwa_diag_history = []

    t0 = time.time()
    for step in range(config["n_steps"] + 1):
        if step % config["eval_interval"] == 0:
            val_loss = estimate_val_loss(
                model, train_data, val_data,
                config["eval_iters"], config["seq_len"], config["batch"], device
            )
            val_bpc = val_loss / math.log(2)
            val_bpc_history.append({"step": step, "val_bpc": round(val_bpc, 6)})

            if act_type == "cwa":
                model.eval()
                xb, _ = get_batch("train", train_data, val_data, config["seq_len"], config["batch"], device)
                with torch.no_grad():
                    model(xb)
                diag = model.get_cwa_diagnostics()
                diag["step"] = step
                cwa_diag_history.append({k: round(float(v), 6) if k != "step" else v for k, v in diag.items()})
                model.train()
                logger.info(
                    f"step={step:5d} | {act_type} seed={seed} | val_bpc={val_bpc:.4f} | "
                    f"J={diag.get('J', 0):.3f} Js\u0304={diag.get('J_s_bar', 0):.3f} "
                    f"|x+Jm*|={diag.get('mean_act_mag', 0):.3f}"
                )
            else:
                logger.info(f"step={step:5d} | {act_type} seed={seed} | val_bpc={val_bpc:.4f}")

        if step == config["n_steps"]:
            break

        lr = get_lr(step, config["n_steps"], config["lr"], config["warmup"])
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        xb, yb = get_batch("train", train_data, val_data, config["seq_len"], config["batch"], device)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    elapsed = time.time() - t0
    final_bpc = val_bpc_history[-1]["val_bpc"]
    logger.info(f"DONE {act_type} seed={seed}: val_bpc={final_bpc:.4f} ({elapsed:.1f}s)")

    # Free GPU memory
    del model, optimizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_bpc_final": final_bpc,
        "val_bpc_history": val_bpc_history,
        "cwa_diag_history": cwa_diag_history,
        "elapsed_s": round(elapsed, 2),
    }

## Run All Activations

Trains one model per (activation, seed) combination using the config parameters above.
With demo config (n_steps=100, n_embd=64, n_layer=2) this runs in ~1-2 minutes on CPU.

In [ ]:
CONFIG = {
    "n_embd": N_EMBD,
    "n_head": N_HEAD,
    "n_layer": N_LAYER,
    "seq_len": SEQ_LEN,
    "batch": BATCH,
    "lr": LR,
    "n_steps": N_STEPS,
    "warmup": WARMUP,
    "eval_interval": EVAL_INTERVAL,
    "eval_iters": EVAL_ITERS,
}
logger.info(f"Config: {CONFIG}")

results: dict[str, dict] = {}
for act in ACTIVATIONS:
    for seed in SEEDS:
        key = f"{act}_seed{seed}"
        logger.info(f"\n=== {key} ===")
        results[key] = train_run(act, seed, vocab_size, train_data, val_data, CONFIG, DEVICE)

## Results & Visualization

**Left**: Live demo training curves (BPC vs step) for the small-scale demo run.

**Right**: Pre-computed reference results from `mini_demo_data.json` (full-scale run, seed=42).

The pre-computed results show CWA achieving BPC comparable to GELU and both outperforming SELU.
The CWA diagnostic panel confirms the sech² saturation mechanism described in the hypothesis.

In [ ]:
# ── Parse pre-computed reference data from mini_demo_data.json ────────────────
ref_trajectories = {}
ref_cwa_diag = {}

for ex in data['datasets'][0]['examples']:
    inp = json.loads(ex['input'])
    out = json.loads(ex['output'])
    act = inp['activation']
    step = inp['eval_step']
    ref_trajectories.setdefault(act, []).append((step, out['val_bpc']))
    if act == 'cwa' and out.get('cwa_J') is not None:
        ref_cwa_diag[step] = {
            'J': out['cwa_J'],
            'J_s_bar': out['cwa_J_s_bar'],
            'mean_act_mag': out['cwa_mean_act_mag'],
            'mean_sech2': out['cwa_mean_sech2'],
        }

for act in ref_trajectories:
    ref_trajectories[act].sort()

# ── Print summary table ────────────────────────────────────────────────────────
print("=" * 60)
print("LIVE DEMO RESULTS (small config)")
print("=" * 60)
print(f"{'Activation':<12} {'Seed':<6} {'Final BPC':<12} {'Time (s)'}")
print("-" * 60)
for act in ACTIVATIONS:
    for seed in SEEDS:
        key = f"{act}_seed{seed}"
        if key in results:
            r = results[key]
            print(f"{act:<12} {seed:<6} {r['val_bpc_final']:<12.4f} {r['elapsed_s']:.1f}")

print()
print("=" * 60)
print("PRE-COMPUTED REFERENCE (full run, seed=42, from mini_demo_data.json)")
print("=" * 60)
meta = data['metadata']
print(f"Config: n_embd={meta['config']['n_embd']}, n_layer={meta['config']['n_layer']}, n_steps={meta['config']['n_steps']}")
print(f"{'Activation':<12} {'Final BPC':<12} {'Mean BPC':<12} {'Std BPC'}")
print("-" * 60)
for act, stats in meta['act_summary'].items():
    final_bpc = ref_trajectories[act][-1][1] if act in ref_trajectories else 'N/A'
    print(f"{act:<12} {final_bpc if isinstance(final_bpc, str) else f'{final_bpc:.4f}':<12} {stats['mean_val_bpc']:<12.4f} {stats['std_val_bpc']:.6f}")

print()
cwa_traj = meta.get('cwa_trajectory_summary', {})
if cwa_traj:
    print(f"CWA sech² saturation confirmed: {cwa_traj.get('sech2_saturation_confirmed')}")
    print(f"  J_s_bar: step_0={cwa_traj.get('step_0_J_s_bar'):.4f} → final={cwa_traj.get('step_final_J_s_bar'):.4f}")
    print(f"  mean_act_mag: step_0={cwa_traj.get('step_0_mean_act_mag'):.4f} → final={cwa_traj.get('step_final_mean_act_mag'):.4f}")
    print(f"  J: step_0={cwa_traj.get('step_0_J'):.4f} → final={cwa_traj.get('step_final_J'):.6f}")

# ── Figures ────────────────────────────────────────────────────────────────────
COLORS = {"gelu": "#2196F3", "selu": "#FF9800", "cwa": "#4CAF50"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("CWA vs SELU vs GELU: Char-GPT on Tiny Shakespeare", fontsize=13, fontweight='bold')

# Panel 1: Live demo BPC curves
ax = axes[0]
for act in ACTIVATIONS:
    for seed in SEEDS:
        key = f"{act}_seed{seed}"
        if key in results:
            hist = results[key]['val_bpc_history']
            steps = [h['step'] for h in hist]
            bpcs = [h['val_bpc'] for h in hist]
            ax.plot(steps, bpcs, color=COLORS.get(act, 'gray'), label=act, linewidth=2)
ax.set_xlabel("Step")
ax.set_ylabel("Val BPC")
ax.set_title(f"Live Demo (n_steps={N_STEPS}, n_embd={N_EMBD})")
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Pre-computed reference BPC curves
ax = axes[1]
for act, traj in ref_trajectories.items():
    steps, bpcs = zip(*traj)
    ax.plot(steps, bpcs, color=COLORS.get(act, 'gray'), label=act, linewidth=2, marker='o', markersize=3)
ax.set_xlabel("Step")
ax.set_ylabel("Val BPC")
ax.set_title(f"Pre-computed Reference (seed=42, n_embd={meta['config']['n_embd']})")
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: CWA diagnostics from reference data
ax = axes[2]
if ref_cwa_diag:
    diag_steps = sorted(ref_cwa_diag.keys())
    j_vals = [ref_cwa_diag[s]['J'] for s in diag_steps]
    js_vals = [ref_cwa_diag[s]['J_s_bar'] for s in diag_steps]
    mag_vals = [ref_cwa_diag[s]['mean_act_mag'] for s in diag_steps]
    ax2 = ax.twinx()
    l1, = ax.plot(diag_steps, j_vals, 'g-o', markersize=4, label='J (coupling)', linewidth=2)
    l2, = ax.plot(diag_steps, js_vals, 'b-s', markersize=4, label='J·s̄ (susceptibility)', linewidth=2)
    l3, = ax2.plot(diag_steps, mag_vals, 'r--^', markersize=4, label='|x+Jm*| (act mag)', linewidth=2)
    ax.set_xlabel("Step")
    ax.set_ylabel("J / J·s̄")
    ax2.set_ylabel("|activation magnitude|")
    ax.set_title("CWA Diagnostics (pre-computed, seed=42)")
    lines = [l1, l2, l3]
    ax.legend(lines, [l.get_label() for l in lines], loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No CWA diagnostics\navailable', ha='center', va='center',
            transform=ax.transAxes, fontsize=12)
    ax.set_title("CWA Diagnostics")

plt.tight_layout()
plt.savefig("cwa_demo_results.png", dpi=100, bbox_inches='tight')
plt.show()
print("\nFigure saved to cwa_demo_results.png")